# 1. Imports

## 1.1. Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import TimeSeriesSplit
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.metrics import (mean_absolute_error, mean_squared_error)



## 1.2. Loading Data

In [2]:
df_sell_in = pd.read_excel('Business_Case_Data_Set.xlsx', sheet_name='Table 2 - Sell In')
df_external_var = pd.read_excel('Business_Case_Data_Set.xlsx', sheet_name = 'Table 1 - External Variables')
df_data_dic = pd.read_excel('Business_Case_Data_Set.xlsx', sheet_name = 'Table 0 - Data Dictionary')

# 2. Data Description

In [3]:
#Join the dfs:

df = pd.merge(df_external_var, df_sell_in, how='left', on=['Week', 'Product', 'Channel'])
df.head(5)

,Week,Product,Channel,Price_per_kg_USD,Numeric_Distribution,Weighted_Distribution,Advertising_Investment_USD,Promotion_Investment_USD,Promotion_Type,Sell_In_Tons
0,2023-07-03,Natural Juice 1L,Supermarkets,2.47,81.3,86.1,7592,1968,NaN,5.592
1,2023-07-03,Natural Juice 1L,Traditional,2.40,57.2,44.0,2962,1818,Bundle,3.278
2,2023-07-03,Natural Juice 1L,E-commerce,2.58,52.1,61.4,6245,3719,Discount,0.410
3,2023-07-03,Flavored Water 500ml,Supermarkets,1.86,68.4,74.6,7711,1795,Discount,3.531
4,2023-07-03,Flavored Water 500ml,Traditional,1.86,56.2,53.6,3409,2465,Discount,2.134


In [4]:
#Separate the traning and predict df
df_pred = df.loc[df['Sell_In_Tons'].isna()].copy()
df_train = df.loc[~df['Sell_In_Tons'].isna()].copy()

In [5]:
# Verify the sell in df
df_sell_in.head(10)

,Week,Product,Channel,Sell_In_Tons
0,2023-07-03,Natural Juice 1L,Supermarkets,5.592
1,2023-07-03,Natural Juice 1L,Traditional,3.278
2,2023-07-03,Natural Juice 1L,E-commerce,0.410
3,2023-07-03,Flavored Water 500ml,Supermarkets,3.531
4,2023-07-03,Flavored Water 500ml,Traditional,2.134
5,2023-07-03,Flavored Water 500ml,E-commerce,0.250
6,2023-07-03,Energy Drink 350ml,Supermarkets,3.350
7,2023-07-03,Energy Drink 350ml,Traditional,2.952
8,2023-07-03,Energy Drink 350ml,E-commerce,0.555
9,2023-07-03,Whole Grain Crackers 200g,Supermarkets,1.325


* Every line is the Sell In in tons, by week, per product in each Channel

In [6]:
# Verify the External Values df
df_external_var.head(5)

,Week,Product,Channel,Price_per_kg_USD,Numeric_Distribution,Weighted_Distribution,Advertising_Investment_USD,Promotion_Investment_USD,Promotion_Type
0,2023-07-03,Natural Juice 1L,Supermarkets,2.47,81.3,86.1,7592,1968,NaN
1,2023-07-03,Natural Juice 1L,Traditional,2.40,57.2,44.0,2962,1818,Bundle
2,2023-07-03,Natural Juice 1L,E-commerce,2.58,52.1,61.4,6245,3719,Discount
3,2023-07-03,Flavored Water 500ml,Supermarkets,1.86,68.4,74.6,7711,1795,Discount
4,2023-07-03,Flavored Water 500ml,Traditional,1.86,56.2,53.6,3409,2465,Discount


In [7]:
# Check the Variables definition and type:
df_data_dic

,Table,Column,Description,Data_Type
0,Table 1,Week,Week start date (Monday),Date
1,Table 1,Product,Product name,String
2,Table 1,Channel,Sales channel,String
3,Table 1,Price_per_kg_USD,Price per kilogram in USD,Float
4,Table 1,Numeric_Distribution,Numeric distribution (%),Float
5,Table 1,Weighted_Distribution,Weighted distribution (%),Float
6,Table 1,Advertising_Investment_USD,Advertising investment in USD,Float
7,Table 1,Promotion_Investment_USD,Promotion investment in USD,Float
8,Table 1,Promotion_Type,"Type of promotion (None, Discount, BOGO, Bundle)",String
9,Table 2,Week,Week start date (Monday),Date


In [8]:
print('Number of Rows of External Variables df: {}'.format(df_external_var.shape[0]))
print('Number of Cols of External Variables df: {}'.format(df_external_var.shape[1]))

Number of Rows of External Variables df: 3510
Number of Cols of External Variables df: 9


In [9]:
# Columns types information:
df_external_var.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3510 entries, 0 to 3509
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Week                        3510 non-null   object 
 1   Product                     3510 non-null   object 
 2   Channel                     3510 non-null   object 
 3   Price_per_kg_USD            3510 non-null   float64
 4   Numeric_Distribution        3510 non-null   float64
 5   Weighted_Distribution       3510 non-null   float64
 6   Advertising_Investment_USD  3510 non-null   int64  
 7   Promotion_Investment_USD    3510 non-null   int64  
 8   Promotion_Type              1910 non-null   object 
dtypes: float64(3), int64(2), object(4)
memory usage: 246.9+ KB


## 1.1. Sell In/Train Description

In [10]:
print('Number of Rows of Sell In df: {}'.format(df_sell_in.shape[0]))
print('Number of Cols of Sell In df: {}'.format(df_sell_in.shape[1]))

Number of Rows of Sell In df: 2340
Number of Cols of Sell In df: 4


In [11]:
print('Number of Rows of train df: {}'.format(df_train.shape[0]))
print('Number of Cols of train df: {}'.format(df_train.shape[1]))

Number of Rows of train df: 2340
Number of Cols of train df: 10


In [12]:
# Columns types information:
df_sell_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2340 entries, 0 to 2339
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Week          2340 non-null   object 
 1   Product       2340 non-null   object 
 2   Channel       2340 non-null   object 
 3   Sell_In_Tons  2340 non-null   float64
dtypes: float64(1), object(3)
memory usage: 73.3+ KB


* The date period is in the following format: YYYY-MM-DD, and represents the first day of the week (monday)

In [13]:
#Transform Week in Date Time
df_sell_in['Week'] = pd.to_datetime(df_sell_in['Week'])
df_external_var['Week'] = pd.to_datetime(df_external_var['Week'])
df_train['Week'] = pd.to_datetime(df_train['Week'])
df_pred['Week'] = pd.to_datetime(df_pred['Week'])

In [14]:
df_sell_in.head(5)

,Week,Product,Channel,Sell_In_Tons
0,2023-07-03,Natural Juice 1L,Supermarkets,5.592
1,2023-07-03,Natural Juice 1L,Traditional,3.278
2,2023-07-03,Natural Juice 1L,E-commerce,0.410
3,2023-07-03,Flavored Water 500ml,Supermarkets,3.531
4,2023-07-03,Flavored Water 500ml,Traditional,2.134


In [15]:
week_count = df_sell_in['Week'].nunique()
week_first = df_sell_in['Week'].min()
week_last = df_sell_in['Week'].max()
week_diff =  ((week_last.year - week_first.year) * 12 + (week_last.month - week_first.month))

print(f'The historic data starts in {week_first} and ends in {week_last}, with a total of {week_count} weeks and {week_diff} months.')

The historic data starts in 2023-07-03 00:00:00 and ends in 2026-06-22 00:00:00, with a total of 156 weeks and 35 months.


In [16]:
#Check NA
df_sell_in.isna().sum()

Week            0
Product         0
Channel         0
Sell_In_Tons    0
dtype: int64

In [17]:
#Check NA
df_train.isna().sum()

Week                             0
Product                          0
Channel                          0
Price_per_kg_USD                 0
Numeric_Distribution             0
Weighted_Distribution            0
Advertising_Investment_USD       0
Promotion_Investment_USD         0
Promotion_Type                1047
Sell_In_Tons                     0
dtype: int64

In [18]:
#Check NA
df_external_var.isna().sum()

Week                             0
Product                          0
Channel                          0
Price_per_kg_USD                 0
Numeric_Distribution             0
Weighted_Distribution            0
Advertising_Investment_USD       0
Promotion_Investment_USD         0
Promotion_Type                1600
dtype: int64

## 1.2. Pred df Description

In [19]:
week_count = df_external_var['Week'].nunique()
week_first = df_external_var['Week'].min()
week_last = df_external_var['Week'].max()
week_diff =  ((week_last.year - week_first.year) * 12 + (week_last.month - week_first.month))

print(f'The historic data starts in {week_first} and ends in {week_last}, with a total of {week_count} weeks and {week_diff} months.')

The historic data starts in 2023-07-03 00:00:00 and ends in 2027-12-20 00:00:00, with a total of 234 weeks and 53 months.


In [20]:
week_count = df_pred['Week'].nunique()
week_first = df_pred['Week'].min()
week_last = df_pred['Week'].max()
week_diff =  ((week_last.year - week_first.year) * 12 + (week_last.month - week_first.month))

print(f'The historic data starts in {week_first} and ends in {week_last}, with a total of {week_count} weeks and {week_diff} months.')

The historic data starts in 2026-06-29 00:00:00 and ends in 2027-12-20 00:00:00, with a total of 78 weeks and 18 months.


## 2.1. Fill Na

In [21]:
df.isna().sum()

Week                             0
Product                          0
Channel                          0
Price_per_kg_USD                 0
Numeric_Distribution             0
Weighted_Distribution            0
Advertising_Investment_USD       0
Promotion_Investment_USD         0
Promotion_Type                1600
Sell_In_Tons                  1170
dtype: int64

In [22]:
df_train['Promotion_Type'] = df_train['Promotion_Type'].fillna('unidentified')
df_pred['Promotion_Type'] = df_pred['Promotion_Type'].fillna('unidentified')

# 5. Data Transformation

In [23]:
df_train['Promotion_Type'].unique()

array(['unidentified', 'Bundle', 'Discount', 'BOGO'], dtype=object)

In [24]:
# Promotion_Type encoding
encoding = {
    'unidentified':0,
    'Discount':1,
    'BOGO':2,
    'Bundle':3
}

df_train['Promotion_Type'] = df_train['Promotion_Type'].map(encoding)

## 5.2. Deal with duplicates

In [25]:
df_train[df_train[['Week', 'Product', 'Channel']].duplicated(keep=False)]

,Week,Product,Channel,Price_per_kg_USD,Numeric_Distribution,Weighted_Distribution,Advertising_Investment_USD,Promotion_Investment_USD,Promotion_Type,Sell_In_Tons


In [26]:
df_train = df_train.drop_duplicates(subset=['Week', 'Product', 'Channel']).reset_index(drop=True)

# 4. Features Engineering

* **Lag_1w:** Sell-In volume lagged by 1 week
* **Lag_4w:**	Sell-In volume lagged by 4 weeks
* **Rolling_Mean_4w:** Rolling mean of Sell-In over last 4 weeks
* **Rolling_Std_4w:**	Rolling standard deviation of Sell-In over last 4 weeks
* **Price_Change_Pct:** Week-over-week percentage change in Price per kg
* **Promo_Flag:**	Binary flag: 1 if Promotion_Type != None, else 0
* **Month_Sin:** Sine transformation of month number (cyclical encoding)
* **Month_Cos:** Cosine transformation of month number (cyclical encoding)
* **Holiday_Flag:** Binary flag: 1 if week contains a national holiday, else 0
* **Interaction_Price_Promo:** Interaction term: Price_per_kg_USD * Promo_Flag (captures joint effect of price and promotion)


In [27]:
# Lag_1w and Lag_4w
df_train = df_train.sort_values(["Channel", "Product", "Week"]).reset_index(drop=True)

df_train["Lag_1w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(1))
df_train["Lag_2w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(2))
df_train["Lag_3w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(3))
df_train["Lag_4w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(4))


df_pred = df_pred.sort_values(["Channel", "Product", "Week"]).reset_index(drop=True)

df_pred["Lag_1w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(1))
df_pred["Lag_2w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(2))
df_pred["Lag_3w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(3))
df_pred["Lag_4w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(4))

In [28]:
# Rolling_Mean_4w

df_train['Rolling_Mean_4w'] = df_train[['Lag_1w', 'Lag_2w', 'Lag_3w', 'Lag_4w']].mean(axis=1)

In [29]:
# Rolling_Std_4w

df_train['Rolling_Std_4w'] = df_train[['Lag_1w', 'Lag_2w', 'Lag_3w', 'Lag_4w']].std(axis=1)

In [30]:
# Price_Change_Pct
df_train = df_train.sort_values(["Channel", "Product", "Week"])

df_train["Price_Change_Pct"] = (df_train.groupby(["Channel", "Product"])["Price_per_kg_USD"].pct_change())

In [31]:
# Promo_Flag
df_train['Promo_Flag'] = np.where(df_train['Promotion_Type'] != 0, 1, 0)

In [32]:
# Month_Sin and Month_Cos
df_train["Month"] = df_train["Week"].dt.month
df_train["Month_Sin"] = np.sin(2*np.pi*df_train["Month"]/12)
df_train["Month_cos"] = np.cos(2*np.pi*df_train["Month"]/12)

In [33]:
# Holiday_Flag


In [34]:
# CREATE DF WITH HOLLIDAYS IN ECUADOR
years = range(2023, 2027)
country = "EC"

all_holidays = []

for year in years:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country}"
    response = requests.get(url)
    response.raise_for_status()

    data = response.json()  # lista de dicionários

    for item in data:
        item["year"] = year
        all_holidays.append(item)

df_holidays = pd.DataFrame(all_holidays)
df_holidays["date"] = pd.to_datetime(df_holidays["date"])

In [35]:
#CREATE DICTIONARY OF WEEK
df_calendar = pd.DataFrame({
    "Date": pd.date_range(
        start="2023-07-03",
        end="2027-12-20",
        freq="D"
    )
})

df_calendar["Week_Number"] = (
    ((df_calendar["Date"] - pd.Timestamp("2023-07-03")).dt.days // 7) + 1
)

df_calendar["Week_Start"] = (
    pd.Timestamp("2023-07-03") +
    pd.to_timedelta((df_calendar["Week_Number"] - 1) * 7, unit="D")
)

df_calendar["Day_Name"] = df_calendar["Date"].dt.day_name()

In [36]:
#Identify the week number in both dfs (holidays and train)
df_train = df_train.merge(df_calendar[['Date', 'Week_Number']], how='left', left_on='Week', right_on='Date').drop(columns='Date')
df_holidays = df_holidays.merge(df_calendar[['Date', 'Week_Number']], how='left', left_on='date', right_on='Date').drop(columns='Date')
df_holidays['Week_Number'] = df_holidays['Week_Number'].astype("Int64")
df_holidays['Holiday'] = 1


In [37]:
df_holidays = (df_holidays.groupby("Week_Number", as_index=False)["Holiday"].max())

In [38]:
#Identify the holiday weeks in the df_train

df_train = df_train.merge(df_holidays[['Week_Number', 'Holiday']], how='left', on='Week_Number')
df_train['Holiday'] = df_train['Holiday'].fillna(0).astype("Int64")
df_train.rename(columns={"Holiday": "Holiday_Flag"}, inplace=True)

In [39]:
df_pred = df_pred.merge(df_calendar[['Date', 'Week_Number']], how='left', left_on='Week', right_on='Date').drop(columns='Date')


In [40]:
#Interaction_Price_Promo
df_train['Interaction_Price_Promo'] = df_train['Price_per_kg_USD'] * df_train['Promo_Flag']

# 5. Exploratory Data Analysis

* Quanto foi vendido por produto e channel ao longo do tempo ? (Gráfico de linhas)
* 

# 6. Product/Channel Filter

In [41]:
keys = (df_train[['Product', 'Channel']]).drop_duplicates().reset_index(drop=True)
keys

,Product,Channel
0,Cereal Bar 50g,E-commerce
1,Energy Drink 350ml,E-commerce
2,Flavored Water 500ml,E-commerce
3,Natural Juice 1L,E-commerce
4,Whole Grain Crackers 200g,E-commerce
5,Cereal Bar 50g,Supermarkets
6,Energy Drink 350ml,Supermarkets
7,Flavored Water 500ml,Supermarkets
8,Natural Juice 1L,Supermarkets
9,Whole Grain Crackers 200g,Supermarkets


In [42]:
#df_train2 = df_train.copy()

In [43]:
#df_train = df_train2[(df_train2['Product'] == 'Cereal Bar 50g') & (df_train2['Channel'] == 'Traditional')]
#df_train = df_train.reset_index(drop=True)

# 7. Features Selection

In [44]:
df_train.columns

Index(['Week', 'Product', 'Channel', 'Price_per_kg_USD',
       'Numeric_Distribution', 'Weighted_Distribution',
       'Advertising_Investment_USD', 'Promotion_Investment_USD',
       'Promotion_Type', 'Sell_In_Tons', 'Lag_1w', 'Lag_2w', 'Lag_3w',
       'Lag_4w', 'Rolling_Mean_4w', 'Rolling_Std_4w', 'Price_Change_Pct',
       'Promo_Flag', 'Month', 'Month_Sin', 'Month_cos', 'Week_Number',
       'Holiday_Flag', 'Interaction_Price_Promo'],
      dtype='object')

In [45]:
df_train = df_train.drop(columns=['Lag_1w','Lag_2w', 'Lag_3w','Lag_4w', 'Month', 'Week_Number', 'Rolling_Std_4w', 'Rolling_Mean_4w'])

# 8. Machine Learnin Modeling 

* I decided using the method of the rolling backtest, using 52 weeks (1 year) as window, i.e. a whole year.

In [46]:

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
 

## 8.1. SETTINGS

In [47]:

# ---------------------------------------------------------
# CONFIGURAÇÕES — ajuste conforme seu projeto
# ---------------------------------------------------------
TARGET = "Sell_In_Tons"
DATE_COL = "Week"          # coluna de data em df_train (datetime, formato YYYY-MM-DD)
HORIZON = 78                # semanas a validar (igual ao horizonte real de previsão)
GROUP_COLS = ["Product", "Channel"]   # colunas que identificam cada série/combinação
LAGS = [1, 4]
ROLL_WINDOW = 4
 
# Hiperparâmetros já selecionados
LGBM_PARAMS = {
    "n_estimators": 500,
    "learning_rate": 0.1,
    "max_depth": 12,
    "num_leaves": 256,
    "min_child_samples": 5,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "min_split_gain": 0.0,
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "mape",
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
}
 

## 8.2. Functions

In [48]:

# ---------------------------------------------------------
# 1. Função para criar as features dependentes do target
#    (mesma lógica usada no treino e, passo a passo, na validação)
# ---------------------------------------------------------
def add_target_features(df, target_col=TARGET, lags=LAGS, window=ROLL_WINDOW):
    df = df.copy()
    for l in lags:
        df[f"{target_col}_lag{l}"] = df[target_col].shift(l)
    df[f"{target_col}_roll_mean{window}"] = df[target_col].shift(1).rolling(window).mean()
    df[f"{target_col}_roll_std{window}"] = df[target_col].shift(1).rolling(window).std(ddof=1)
    return df
 
 
# ---------------------------------------------------------
# 2. Validação recursiva para UMA combinação (uma série já filtrada)
# ---------------------------------------------------------
def run_validation(df_sub, group_cols=GROUP_COLS, verbose=True):
    df_sub = df_sub.sort_values(DATE_COL).reset_index(drop=True)
 
    # exclui data, target e as colunas identificadoras (Product/Channel) das features,
    # já que dentro de cada combinação elas são constantes
    excluded_cols = [DATE_COL, TARGET] + [c for c in group_cols if c in df_sub.columns]
    feature_cols_base = [c for c in df_sub.columns if c not in excluded_cols]
    target_feature_cols = (
        [f"{TARGET}_lag{l}" for l in LAGS]
        + [f"{TARGET}_roll_mean{ROLL_WINDOW}", f"{TARGET}_roll_std{ROLL_WINDOW}"]
    )
    feature_cols = feature_cols_base + target_feature_cols
 
    # -----------------------------------------------------
    # Split temporal
    # -----------------------------------------------------
    n_total = len(df_sub)
    n_train = n_total - HORIZON
    if n_train <= max(LAGS + [ROLL_WINDOW]):
        raise ValueError(
            f"Histórico insuficiente para essa combinação "
            f"(n_total={n_total}, necessário > {HORIZON + max(LAGS + [ROLL_WINDOW])})."
        )
 
    df_hist = df_sub.iloc[:n_train].copy()
    df_valid = df_sub.iloc[n_train:].reset_index(drop=True).copy()
 
    if verbose:
        print(f"Treino: {len(df_hist)} semanas | Validação: {len(df_valid)} semanas")
 
    # -----------------------------------------------------
    # Dataset de treino (lags/rolling calculados com dados reais)
    # -----------------------------------------------------
    df_hist_feat = add_target_features(df_hist).dropna().reset_index(drop=True)
 
    X_train = df_hist_feat[feature_cols]
    y_train = df_hist_feat[TARGET]
 
    # -----------------------------------------------------
    # Treinamento do modelo
    # -----------------------------------------------------
    model = lgb.LGBMRegressor(**LGBM_PARAMS)
    model.fit(X_train, y_train)
 
    # -----------------------------------------------------
    # Validação recursiva (walk-forward)
    # -----------------------------------------------------
    history = df_hist[TARGET].tolist()  # série real conhecida até o início da validação
    preds = []
 
    for i in range(len(df_valid)):
        row = df_valid.loc[[i]].copy()
 
        lag1 = history[-1]
        lag4 = history[-4]
        roll_mean = np.mean(history[-ROLL_WINDOW:])
        roll_std = np.std(history[-ROLL_WINDOW:], ddof=1)
 
        row[f"{TARGET}_lag1"] = lag1
        row[f"{TARGET}_lag4"] = lag4
        row[f"{TARGET}_roll_mean{ROLL_WINDOW}"] = roll_mean
        row[f"{TARGET}_roll_std{ROLL_WINDOW}"] = roll_std
 
        X_step = row[feature_cols]
        y_hat = model.predict(X_step)[0]
 
        preds.append(y_hat)
        history.append(y_hat)  # usa a PREVISÃO, não o valor real -> simula produção
 
    # -----------------------------------------------------
    # Avaliação
    # -----------------------------------------------------
    y_true = df_valid[TARGET].values
    y_pred = np.array(preds)
 
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
 
    # MAPE — atenção: indefinido/instável se y_true tiver valores 0 ou muito próximos de 0.
    # Se o seu SI puder ter zeros, considere usar WMAPE como complemento (mais robusto).
    mape = np.mean(np.abs((y_true - y_pred) / y_true))
    wmape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))
 
    if verbose:
        print(f"MAPE:  {mape:.2%}")
 
    df_result = df_valid[[DATE_COL, TARGET]].copy()
    df_result["pred"] = y_pred
    df_result["abs_pct_error"] = np.abs((df_result[TARGET] - df_result["pred"]) / df_result[TARGET])
 
    return model, df_result, {"mape": mape}
 
 
# ---------------------------------------------------------
# 3. Loop por todas as combinações de Product x Channel
# ---------------------------------------------------------
def run_validation_all(df_train, group_cols=GROUP_COLS, verbose=False):
    """
    Roda run_validation individualmente para cada combinação de group_cols
    (ex: Product x Channel) e consolida os resultados em uma tabela-resumo.
 
    Retorna:
        df_summary: DataFrame com uma linha por combinação e as métricas (MAPE, WMAPE, MAE, RMSE)
        detailed_results: dict {combinação: {"model":..., "df_result":...}} para inspeção posterior
    """
    combinations = df_train[group_cols].drop_duplicates().reset_index(drop=True)
 
    summary_rows = []
    detailed_results = {}
 
    for idx, combo in combinations.iterrows():
        filtro = np.all(
            [df_train[col] == combo[col] for col in group_cols], axis=0
        )
        df_sub = df_train[filtro].copy()
 
        combo_key = tuple(combo[col] for col in group_cols)
        combo_label = " | ".join(f"{col}={combo[col]}" for col in group_cols)
 
        print(f"\n=== Validando: {combo_label} ===")
 
        row = {col: combo[col] for col in group_cols}
 
        try:
            model, df_result, metrics = run_validation(df_sub, group_cols=group_cols, verbose=verbose)
            row.update(metrics)
            row["n_semanas"] = len(df_sub)
            row["status"] = "ok"
            detailed_results[combo_key] = {"model": model, "df_result": df_result}
            print(f"MAPE: {metrics['mape']:.2%}")
        except Exception as e:
            row.update({"mape": np.nan})
            row["n_semanas"] = len(df_sub)
            row["status"] = f"erro: {e}"
            print(f"[ERRO] {combo_label} -> {e}")
 
        summary_rows.append(row)
 
    df_summary = pd.DataFrame(summary_rows)
    df_summary = df_summary.sort_values("mape", ascending=False, na_position="last").reset_index(drop=True)
 
    return df_summary, detailed_results
 
 
def plot_validation_results(df_result, title_suffix="", target_col=TARGET, date_col=DATE_COL):
    """
    Gera dois gráficos para uma combinação específica:
      1. Real vs. Previsto ao longo das semanas de validação.
      2. Erro percentual absoluto (%) por semana, para identificar onde o modelo
         degrada mais (ex: acúmulo de erro recursivo nas semanas finais do horizonte).
    """
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
 
    axes[0].plot(df_result[date_col], df_result[target_col], label="Real", marker="o", markersize=3)
    axes[0].plot(df_result[date_col], df_result["pred"], label="Previsto", marker="o", markersize=3)
    axes[0].set_title(f"SI: Real vs. Previsto {title_suffix}")
    axes[0].set_ylabel("SI")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
 
    erro_pct = df_result["abs_pct_error"] * 100
    axes[1].bar(df_result[date_col], erro_pct, color="indianred", alpha=0.7)
    axes[1].axhline(erro_pct.mean(), color="black", linestyle="--", linewidth=1,
                     label=f"MAPE médio: {erro_pct.mean():.1f}%")
    axes[1].set_title("Erro percentual absoluto por semana")
    axes[1].set_ylabel("Erro (%)")
    axes[1].set_xlabel(date_col)
    axes[1].legend()
    axes[1].grid(alpha=0.3)
 
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
 
    return fig
 
 

## 8.2. Training the model with recursive features

In [49]:

# ---------------------------------------------------------
# EXECUÇÃO
# ---------------------------------------------------------
# Assume que df_train já existe no ambiente, agora contendo as colunas
# Product e Channel além de date, SI e demais features exógenas.
 
df_summary, detailed_results = run_validation_all(df_train)
 
print("\n=== Tabela-resumo: MAPE por combinação ===")
print(df_summary)
 
# Exemplo: para inspecionar uma combinação específica em detalhe (gráfico), use:
# combo_key = (nome_produto, nome_canal)
# plot_validation_results(detailed_results[combo_key]["df_result"], title_suffix=str(combo_key))
 


=== Validando: Product=Cereal Bar 50g | Channel=E-commerce ===
MAPE: 12.21%

=== Validando: Product=Energy Drink 350ml | Channel=E-commerce ===
MAPE: 157.87%

=== Validando: Product=Flavored Water 500ml | Channel=E-commerce ===
MAPE: 87.79%

=== Validando: Product=Natural Juice 1L | Channel=E-commerce ===
MAPE: 86.42%

=== Validando: Product=Whole Grain Crackers 200g | Channel=E-commerce ===
MAPE: 15.81%

=== Validando: Product=Cereal Bar 50g | Channel=Supermarkets ===
MAPE: 24.44%

=== Validando: Product=Energy Drink 350ml | Channel=Supermarkets ===
MAPE: 25.85%

=== Validando: Product=Flavored Water 500ml | Channel=Supermarkets ===
MAPE: 30.25%

=== Validando: Product=Natural Juice 1L | Channel=Supermarkets ===
MAPE: 25.55%

=== Validando: Product=Whole Grain Crackers 200g | Channel=Supermarkets ===
MAPE: 28.38%

=== Validando: Product=Cereal Bar 50g | Channel=Traditional ===
MAPE: 37.36%

=== Validando: Product=Energy Drink 350ml | Channel=Traditional ===
MAPE: 22.48%

=== Validand

In [50]:
df_summary

,Product,Channel,mape,n_semanas,status
0,Energy Drink 350ml,E-commerce,1.578665,156,ok
1,Flavored Water 500ml,E-commerce,0.877872,156,ok
2,Natural Juice 1L,E-commerce,0.864240,156,ok
3,Whole Grain Crackers 200g,Traditional,0.479427,156,ok
4,Cereal Bar 50g,Traditional,0.373629,156,ok
5,Natural Juice 1L,Traditional,0.310850,156,ok
6,Flavored Water 500ml,Traditional,0.307805,156,ok
7,Flavored Water 500ml,Supermarkets,0.302483,156,ok
8,Whole Grain Crackers 200g,Supermarkets,0.283790,156,ok
9,Energy Drink 350ml,Supermarkets,0.258539,156,ok


# 9. Hyperparameter Fine Tuning

# 10. Translation and Interpretation of Performance